In [ ]:
# Gohst-Guided TSP Benchmarking Notebook

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter
from scipy.spatial.distance import pdist

class Gohst:
    def __init__(self, position, direction, is_dark=False):
        self.position = np.array(position, dtype=float)
        self.direction = np.array(direction, dtype=float)
        self.is_dark = is_dark
        self.collapsed = False

    def move(self, step_size=1.0):
        self.position += self.direction * step_size

def displace_home_by_ratio(home, S, E):
    x, y = home.astype(int)
    field_x = min(max(x, 0), S.shape[0]-1)
    field_y = min(max(y, 0), S.shape[1]-1)
    symbolic_val = S[field_x, field_y]
    entropy_val = E[field_x, field_y]
    if symbolic_val + entropy_val == 0:
        return home
    direction = np.array([np.cos(symbolic_val), np.sin(entropy_val)])
    new_home = home + direction
    return np.clip(new_home, 0, S.shape[0]-1)

def path_cost(path, G):
    return sum(G[path[i]][path[i+1]]['weight'] for i in range(len(path)-1))

ALPHA = 2.0
BETA = 1.5

def run_gohst_benchmark(NUM_CITIES=50, FIELD_SIZE=100, gamma=0.01, STEPS=50):
    np.random.seed(42)
    cities = np.random.rand(NUM_CITIES, 2) * FIELD_SIZE
    home_city = cities[0]
    city_mass = np.mean(pdist(cities))

    ancestral_home_list = [home_city.copy()]

    symbolic_field = np.zeros((FIELD_SIZE, FIELD_SIZE))
    entropy_field = np.zeros((FIELD_SIZE, FIELD_SIZE))

    home_city = displace_home_by_ratio(home_city, symbolic_field, entropy_field)
    ancestral_home_list.append(home_city.copy())

    NUM_GOHSTS = int(gamma * FIELD_SIZE ** 2)
    projected_gohsts = []
    for i in range(NUM_GOHSTS):
        angle = 2 * np.pi * i / NUM_GOHSTS
        direction = np.array([np.cos(angle), np.sin(angle)])
        projected_gohsts.append(Gohst(home_city.copy(), direction, is_dark=False))
    for i in range(NUM_GOHSTS // 2):
        angle = 2 * np.pi * i / (NUM_GOHSTS // 2)
        direction = np.array([np.cos(angle), np.sin(angle)])
        projected_gohsts.append(Gohst(home_city.copy(), direction, is_dark=True))

    for step in range(STEPS):
        for gohst in projected_gohsts:
            if not gohst.collapsed and step == STEPS // 2:
                x, y = gohst.position.astype(int)
                x = np.clip(x, 0, FIELD_SIZE - 1)
                y = np.clip(y, 0, FIELD_SIZE - 1)
                if gohst.is_dark:
                    entropy_field[x, y] += city_mass * 6
                else:
                    symbolic_field[x, y] += city_mass
                gohst.collapsed = True
            if not gohst.collapsed:
                gohst.move()

    symbolic_field = gaussian_filter(symbolic_field, sigma=3)
    entropy_field = gaussian_filter(entropy_field, sigma=3)

    G = nx.Graph()
    for i in range(NUM_CITIES):
        for j in range(i + 1, NUM_CITIES):
            xi, yi = cities[i]
            xj, yj = cities[j]
            dist = np.linalg.norm(cities[i] - cities[j])
            mid = ((xi + xj) / 2, (yi + yj) / 2)
            mx, my = np.clip(np.round(mid).astype(int), 0, FIELD_SIZE - 1)
            symbolic_val = symbolic_field[mx, my]
            entropy_val = entropy_field[mx, my]
            weight = dist - ALPHA * symbolic_val + BETA * entropy_val
            weight = max(weight, 1e-3)
            G.add_edge(i, j, weight=weight)

    tsp_path = nx.approximation.traveling_salesman_problem(G, cycle=True)
    cost = path_cost(tsp_path, G)

    return cost, tsp_path, cities, home_city, ancestral_home_list, symbolic_field, entropy_field

def plot_gohst_result(cost, tsp_path, cities, home_city, ancestry, symbolic_field=None, entropy_field=None):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(0, np.max(cities[:, 0]) + 10)
    ax.set_ylim(0, np.max(cities[:, 1]) + 10)

    if symbolic_field is not None:
        ax.imshow(symbolic_field.T, origin="lower", cmap="Reds", alpha=0.25, extent=[0, symbolic_field.shape[0], 0, symbolic_field.shape[1]])
    if entropy_field is not None:
        ax.imshow(entropy_field.T, origin="lower", cmap="Blues", alpha=0.2, extent=[0, entropy_field.shape[0], 0, entropy_field.shape[1]])

    ax.scatter(cities[:, 0], cities[:, 1], c="black", s=50, label="City")
    for idx, anc_home in enumerate(ancestry[:-1]):
        ax.scatter(*anc_home, c="gray", s=70, marker="X", label="Black Hole" if idx == 0 else None)

    ax.scatter(*home_city, c="gold", s=100, marker="*", label="Current Home")
    final_node = cities[tsp_path[-2]]
    ax.scatter(*final_node, c="cyan", s=100, marker="o", edgecolors="black", linewidths=1.5, label="White Hole")

    for i in range(len(tsp_path) - 1):
        p1 = cities[tsp_path[i]]
        p2 = cities[tsp_path[i + 1]]
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], c="green", lw=2)

    legend_elements = [
        Line2D([0], [0], color='green', lw=2, label='Gohst-Guided Path'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='black', label='City', markersize=8),
        Line2D([0], [0], marker='*', color='w', markerfacecolor='gold', label='Current Home', markersize=12),
        Line2D([0], [0], marker='X', color='gray', linestyle='None', label='Black Hole (Ancestral Home)', markersize=10),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='cyan', label='White Hole (Final Node)', markersize=10)
    ]

    ax.legend(handles=legend_elements)
    ax.set_title(f"Gohst-Guided TSP with Drift\nCost: {cost:.2f}")
    ax.grid(True)
    plt.show()

# Example Run
cost, path, cities, home, ancestry, S, E = run_gohst_benchmark(NUM_CITIES=75)
print(f"Final Cost: {cost:.2f}")
plot_gohst_result(cost, path, cities, home, ancestry, symbolic_field=S, entropy_field=E)